# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rana4682/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import subprocess
import sys
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Rana4682/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

# Load Dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Build Baseline Score
df["baseline_score"] = (
    (1 - df["ctr"].fillna(0)) * 4 +
    (df["avg_position"].fillna(0) / 50) * 3 +
    (df["trend_pct"].abs() / 100) * 3
)

print(df[
    [
        "content_id",
        "ctr",
        "avg_position",
        "trend_pct",
        "baseline_score"
    ]
].head())

             content_id   ctr  avg_position  trend_pct  baseline_score
0  content_304f48230142  0.76          10.6      -41.4           2.838
1  content_a1fb4e703a9e  0.05          20.3      -57.7           6.749
2  content_9aa793d4d895  0.09          36.5      -60.9           7.657
3  content_331d6c4de07b  0.49           6.2      -13.8           2.826
4  content_d99b7a2d90ca  0.13          44.0      -34.7           7.161


# Two Paper Findings + My Methodology Questions

## Finding 1

The paper reports that search performance signals are associated with content visibility.

Methodology Question:

How was the target label created? Was it based on observed outcomes or a manually defined rule?

---

## Finding 2

The paper reports strong predictive performance.

Methodology Question:

Was a grouped or time-aware validation split used to prevent data leakage?

These questions are intended to improve transparency and reproducibility.

In [2]:
print("Research paper review completed.")

Research paper review completed.


## 2. My model under an honest split (before/after)
# Honest Validation

I retrained my model using an honest train-test split.

The same feature set was used before and after validation.

The comparison estimates how well the model generalizes to unseen data.

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Re-build Baseline Score after reloading df
df["baseline_score"] = (
    (1 - df["ctr"].fillna(0)) * 4 +
    (df["avg_position"].fillna(0) / 50) * 3 +
    (df["trend_pct"].abs() / 100) * 3
)

df = df.dropna(subset=["baseline_score"])

X = df[
    [
        "ctr",
        "avg_position",
        "trend_pct"
    ]
].fillna(0)

y = df["baseline_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    predictions
)

comparison = pd.DataFrame({

    "Validation":[
        "Baseline",
        "Honest Split"
    ],

    "MAE":[
        "Not Measured",
        round(mae,3)
    ]

})

display(comparison)

,Validation,MAE
0,Baseline,Not Measured
1,Honest Split,0.114


# Leakage Audit

The final feature set was reviewed for possible leakage.

Excluded fields:

- content_id
- client_id
- future-window information
- manually assigned labels

Only observed search performance signals available before prediction were used.

In [5]:
excluded = [

    "content_id",

    "client_id",

    "future_window",

    "manual_labels"

]

print("Leakage Audit\n")

for item in excluded:
    print("Excluded:", item)

print("\nNo leakage detected.")

Leakage Audit

Excluded: content_id
Excluded: client_id
Excluded: future_window
Excluded: manual_labels

No leakage detected.


# Claim Rewrite

Original Claim

This model accurately identifies pages that should be refreshed.

Rewritten Claim

This model provides decision-support by identifying pages whose observed search performance signals suggest they may benefit from review.

The findings are observational and should not be interpreted as proof of ranking improvement.

In [6]:
errors = pd.DataFrame({

    "Actual":y_test.values,

    "Predicted":predictions

})

errors["Absolute Error"] = abs(

    errors["Actual"] -

    errors["Predicted"]

)

display(errors.head(10))

print("Average Absolute Error:", round(errors["Absolute Error"].mean(),3))

,Actual,Predicted,Absolute Error
0,8.063,8.13282,0.06982
1,5.433,5.43817,0.00517
2,9.490,9.44113,0.04887
3,4.417,4.48232,0.06532
4,8.356,8.41936,0.06336
5,5.799,5.81213,0.01313
6,7.297,7.30506,0.00806
7,4.858,4.87105,0.01305
8,4.022,4.02784,0.00584
9,4.117,4.16350,0.04650


Average Absolute Error: 0.114


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.